# 기존 앙상블에서 TabICL만 제외한 비교

> 이전 실험 기록: 아래 코드와 출력은 기존 5개 모델 조합에서 TabICL만 제외한 7:3 분할 비교다. RF를 포함한 최신 앙상블 실험과는 다르다. 현재 실행 순서는 [노트북 안내](README.md)를 따르며 두 비교의 수치를 섞지 않는다.

기존 4단계의 **LR + MultinomialNB + ExtraTrees + CatBoost + TabICL**에서 TabICL만 제거한다.
새 RF를 추가하는 실험이 아니며, 원래 조합의 제거 효과를 별도로 확인한다.

- 13개 입력, 행마다 Unknown 총 4개, 마스킹 10세트, Train/Test 그룹 분리를 유지한다.
- 나머지 4개 모델의 최적 파라미터, 5-Fold × 시드 3개, 안쪽 3-Fold, GES 25회, 메타 LR C=0.1을 유지한다.
- 단일 모델 4개와 Soft Voting·GES·Stacking 3개를 비교한다. 분류 임계값은 0.5다.
- 기존 TabICL 포함 결과는 저장된 4단계 출력과 비교한다. 이번에는 TabICL을 다시 학습하거나 불러오지 않는다.
- 기존 배포 모델은 전체 데이터를 학습했으므로 Test 평가에 재사용하지 않는다.

기본 파라미터는 전체 Train으로 이미 선택된 값이다. 결합 학습은 안쪽 폴드에서 분리하지만,
전체 파라미터 탐색까지 포함한 독립적인 nested CV는 아니다. 이미 여러 실험에서 본 Test도 완전히 새로운 최종 검증 표본은 아니다.

## 0. 실행 순서

기존 전처리만 실행하고, 3단계에서 확정한 파라미터를 코드에 명시하여 재사용한다.
없어진 중간 파라미터 파일이나 이전 모델 학습을 다시 실행할 필요가 없다.
원본 CSV 위치는 전처리 노트북의 `SALESLUV_B2B_DATA_PATH` 설정을 따른다.

TabICL·PyTorch를 import하지 않는다. 나머지 모델은 CPU에서 실행한다.

In [1]:
import hashlib
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from IPython import get_ipython
from IPython.display import display
from IPython.utils.capture import capture_output
from sklearn.base import clone, is_classifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 260)


RANDOM_STATE = 1
CLASSIFICATION_THRESHOLD = 0.5
current_dir = Path.cwd().resolve()
if current_dir.name == "notebooks":
    preprocessing_notebook = current_dir / "deal_data_preprocessing.ipynb"
elif current_dir.name == "backend" and (current_dir / "notebooks").is_dir():
    preprocessing_notebook = current_dir / "notebooks" / "deal_data_preprocessing.ipynb"
elif (current_dir / "backend" / "notebooks").is_dir():
    preprocessing_notebook = current_dir / "backend" / "notebooks" / "deal_data_preprocessing.ipynb"
else:
    raise RuntimeError("backend/notebooks 폴더를 찾을 수 없습니다.")

ipython = get_ipython()
assert ipython is not None, "Jupyter 커널에서 실행해야 합니다."
# 원본 데이터 행과 전처리 전체 출력을 이 노트북에 다시 저장하지 않는다.
with capture_output():
    ipython.run_line_magic("run", str(preprocessing_notebook))

X_train_raw = ipython.user_ns["X_train_raw"]
y_train = ipython.user_ns["y_train"]
train_group_ids = ipython.user_ns["train_group_ids"]
X_test_raw_sets = ipython.user_ns["X_test_raw_sets"]
y_test = ipython.user_ns["y_test"]
input_group_ids = ipython.user_ns["input_group_ids"]
MODEL_FEATURE_NAMES = ipython.user_ns["MODEL_FEATURE_NAMES"]
CATEGORY_VALUES = ipython.user_ns["CATEGORY_VALUES"]
SOURCE_SHA256 = ipython.user_ns["SOURCE_SHA256"]

# 입력 순서, 정답 정렬, 마스킹 개수와 Train/Test 그룹 분리를 검증한다.
assert list(X_train_raw.columns) == list(MODEL_FEATURE_NAMES)
assert X_train_raw.index.equals(y_train.index)
assert len(train_group_ids) == len(y_train)
assert X_train_raw.eq("Unknown").sum(axis=1).eq(4).all()
assert set(y_train.unique()) == {0, 1}
assert len(X_test_raw_sets) == 10
for X_test_raw in X_test_raw_sets.values():
    assert list(X_test_raw.columns) == list(MODEL_FEATURE_NAMES)
    assert X_test_raw.index.equals(y_test.index)
    assert X_test_raw.eq("Unknown").sum(axis=1).eq(4).all()
    assert set(train_group_ids).isdisjoint(input_group_ids.loc[X_test_raw.index])

# 기본 모델과 튜닝 후보가 모두 같은 검증 행을 평가하도록 폴드를 한 번만 만든다.
cv5 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_splits = list(cv5.split(X_train_raw, y_train, groups=train_group_ids))
for train_index, valid_index in cv_splits:
    assert set(train_group_ids[train_index]).isdisjoint(set(train_group_ids[valid_index]))
    assert set(y_train.iloc[train_index].unique()) == {0, 1}
    assert set(y_train.iloc[valid_index].unique()) == {0, 1}

train_original_rows = X_train_raw.index.get_level_values("original_row_id").nunique()
print(f"Train: 원본 {train_original_rows}건 → 마스킹 포함 {len(y_train)}행")
print(f"Test: 같은 {len(y_test)}건에 서로 다른 마스킹 {len(X_test_raw_sets)}세트")
print(f"입력 {len(MODEL_FEATURE_NAMES)}개, CV {len(cv_splits)}-Fold, 임계값 0.5")

y_train = np.asarray(y_train, dtype=int)
y_test = np.asarray(y_test, dtype=int)
train_mask_set_labels = X_train_raw.index.get_level_values("mask_set").to_numpy()
OUTER_SEEDS = (1, 11, 21)
OUTER_FOLDS = 5
INNER_FOLDS = 3
GES_STEPS = 25
repo_root = preprocessing_notebook.parents[2]
NO_TABICL_PATH = repo_root / "backend" / "pipeline" / "artifacts" / "deal-no-tabicl-v1.joblib"
assert X_train_raw.shape == (3130, 13)
assert len(np.unique(train_group_ids)) == 138
print(f"반복 평가: {OUTER_FOLDS}-Fold × {len(OUTER_SEEDS)}개 시드")
print(f"안쪽 OOF: {INNER_FOLDS}-Fold, GES: {GES_STEPS}회")

Train: 원본 313건 → 마스킹 포함 3130행
Test: 같은 135건에 서로 다른 마스킹 10세트
입력 13개, CV 5-Fold, 임계값 0.5
반복 평가: 5-Fold × 3개 시드
안쪽 OOF: 3-Fold, GES: 25회


### 해석

- 독립 학습 단위는 마스킹 후 3,130행이 아니라 138개 입력 그룹이다.
- 한 번의 Fold 배정 운에 순위가 좌우되지 않도록 바깥 5-Fold를 시드 3개로 반복한다.
- 동일 원본과 마스킹 변형은 모든 바깥·안쪽 분할에서 같은 Fold에 유지된다.

## 1. 동일한 입력과 확정 파라미터

In [2]:
def training_data_signature() -> str:
    """입력 내용·행 순서·정답·그룹 ID를 함께 기록한다."""
    parts = (
        pd.util.hash_pandas_object(X_train_raw.astype("string"), index=True)
        .to_numpy(dtype=np.uint64)
        .tobytes(),
        np.asarray(y_train, dtype=np.int8).tobytes(),
        np.asarray(train_group_ids, dtype=np.uint64).tobytes(),
    )
    return hashlib.sha256(b"".join(parts)).hexdigest()


# 기존 phase3 출력 및 deal-stacking-lr-v2.json의 best_params와 대조한 값이다.
# 학습된 기존 모델을 불러오지 않고 아래 설정으로 현재 Train만 학습한다.
best_params = {
    "LogisticRegression": {"classifier__C": 0.03, "classifier__class_weight": "balanced"},
    "MultinomialNB": {"classifier__alpha": 75.0, "classifier__fit_prior": False},
    "ExtraTrees": {
        "classifier__max_depth": 8,
        "classifier__max_features": "sqrt",
        "classifier__min_samples_leaf": 6,
        "classifier__n_estimators": 300,
    },
    "CatBoost": {
        "random_strength": 0.5,
        "learning_rate": 0.01,
        "l2_leaf_reg": 7.0,
        "iterations": 200,
        "depth": 4,
    },
}
display(
    pd.DataFrame([{"model": name, "best_params": params} for name, params in best_params.items()])
)

,model,best_params
0,LogisticRegression,"{'classifier__C': 0.03, 'classifier__class_wei..."
1,MultinomialNB,"{'classifier__alpha': 75.0, 'classifier__fit_p..."
2,ExtraTrees,"{'classifier__max_depth': 8, 'classifier__max_..."
3,CatBoost,"{'random_strength': 0.5, 'learning_rate': 0.01..."


### 해석

4개 모델의 설정은 이전과 같고, 바뀌는 것은 TabICL의 제외뿐이다.
전처리에서 원본 파일의 SHA-256을 확인하며, 아래에서 4개 단일 모델의 CV/Test 값이 기존 저장 출력과 재현되는지도 검사한다.
중간 데이터 서명 파일은 남아 있지 않아 그 파일까지 대조한 완전한 재현이라고 주장하지 않는다.

## 2. 모델별 입력 처리를 포함한 최종 기본 모델 구성

In [3]:
def make_one_hot_model(classifier):
    encoder = OneHotEncoder(
        categories=[list(CATEGORY_VALUES[column]) for column in MODEL_FEATURE_NAMES],
        drop="first",
        handle_unknown="error",
        sparse_output=False,
        dtype=np.float32,
    )
    return Pipeline([("onehot", encoder), ("classifier", classifier)])


base_models = {
    "LogisticRegression": make_one_hot_model(
        LogisticRegression(max_iter=3000, solver="lbfgs", random_state=RANDOM_STATE)
    ),
    "MultinomialNB": make_one_hot_model(MultinomialNB()),
    "ExtraTrees": make_one_hot_model(ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=1)),
    "CatBoost": CatBoostClassifier(
        cat_features=tuple(MODEL_FEATURE_NAMES),
        loss_function="Logloss",
        verbose=False,
        allow_writing_files=False,
        random_seed=RANDOM_STATE,
        thread_count=1,
    ),
}
for model_name, model in base_models.items():
    model.set_params(**best_params[model_name])
    assert is_classifier(model)
    assert hasattr(model, "predict_proba")

base_model_names = list(base_models)
assert base_model_names == ["LogisticRegression", "MultinomialNB", "ExtraTrees", "CatBoost"]
display(
    pd.DataFrame(
        [
            {
                "model": name,
                "input": "원본 범주형 13개" if name == "CatBoost" else "모델 내부 원핫 39개",
            }
            for name in base_model_names
        ]
    )
)

,model,input
0,LogisticRegression,모델 내부 원핫 39개
1,MultinomialNB,모델 내부 원핫 39개
2,ExtraTrees,모델 내부 원핫 39개
3,CatBoost,원본 범주형 13개


### 해석

- LR은 규제를 적용한 선형 모델, NB는 범주별 빈도를 활용하는 확률 모델이다.
- ExtraTrees는 무작위 분기를 활용한 트리 앙상블, CatBoost는 원본 범주형 입력을 처리하는 부스팅 모델이다.
- LR·NB·ExtraTrees는 자신의 파이프라인에서 원핫 인코딩하고, CatBoost는 같은 13개 컬럼을 그대로 받는다.
- 학습 방식은 그대로 두고 각 모델이 출력하는 Won 확률만 결합한다.

## 3. 공통 지표와 GES 함수

In [4]:
METRIC_NAMES = (
    "brier",
    "logloss",
    "auc",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "specificity",
    "fpr",
    "tn",
    "fp",
    "fn",
    "tp",
)


def positive_class_probability(estimator, X):
    won_index = list(estimator.classes_).index(1)
    return estimator.predict_proba(X)[:, won_index]


def calculate_metrics(y_true, probability):
    prediction = (probability >= CLASSIFICATION_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp)
    return {
        "brier": brier_score_loss(y_true, probability),
        "logloss": log_loss(y_true, probability, labels=[0, 1]),
        "auc": roc_auc_score(y_true, probability),
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "specificity": specificity,
        "fpr": 1 - specificity,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def calculate_mask_set_averaged_metrics(y_true, probability, mask_set_labels):
    """Train의 10개 마스킹 세트를 각각 평가한 뒤 Test와 같은 방식으로 평균한다."""
    set_metrics = pd.DataFrame(
        [
            calculate_metrics(
                y_true[mask_set_labels == set_name],
                probability[mask_set_labels == set_name],
            )
            for set_name in np.unique(mask_set_labels)
        ]
    )
    return set_metrics.mean().to_dict()


def greedy_ensemble_weights(y_true, probability_matrix, steps=GES_STEPS):
    """OOF Brier를 가장 많이 낮추는 모델을 반복 선택해 비음수 가중치를 만든다."""
    counts = np.zeros(probability_matrix.shape[1], dtype=int)
    probability_sum = np.zeros(len(y_true), dtype=float)
    history = []
    for step in range(steps):
        candidate_brier = [
            brier_score_loss(y_true, (probability_sum + probability_matrix[:, index]) / (step + 1))
            for index in range(probability_matrix.shape[1])
        ]
        selected_index = int(np.argmin(candidate_brier))
        counts[selected_index] += 1
        probability_sum += probability_matrix[:, selected_index]
        history.append(
            {
                "step": step + 1,
                "selected_model": base_model_names[selected_index],
                "brier": candidate_brier[selected_index],
            }
        )
    return counts / counts.sum(), pd.DataFrame(history)


print("공통 지표와 GES 함수를 준비했습니다.")

공통 지표와 GES 함수를 준비했습니다.


### 해석

GES는 같은 모델을 여러 번 선택할 수 있다. 25회 중 선택된 횟수의 비율이 최종 가중치가 되며 선택되지 않은 모델은 자동으로 0이 된다.

## 4. 반복 Group CV

In [5]:
cv_started = perf_counter()
repeat_base_oof = []
repeat_ges_oof = []
repeat_stacking_oof = []
fold_weight_rows = []

for repeat_number, outer_seed in enumerate(OUTER_SEEDS, start=1):
    outer_cv = StratifiedGroupKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=outer_seed)
    outer_splits = list(outer_cv.split(X_train_raw, y_train, groups=train_group_ids))
    base_oof = np.full((len(y_train), len(base_models)), np.nan, dtype=float)
    ges_oof = np.full(len(y_train), np.nan, dtype=float)
    stacking_oof = np.full(len(y_train), np.nan, dtype=float)

    for fold_number, (outer_train, outer_valid) in enumerate(outer_splits, start=1):
        assert set(train_group_ids[outer_train]).isdisjoint(set(train_group_ids[outer_valid]))
        X_outer_train = X_train_raw.iloc[outer_train]
        y_outer_train = y_train[outer_train]
        groups_outer_train = train_group_ids[outer_train]

        inner_cv = StratifiedGroupKFold(
            n_splits=INNER_FOLDS,
            shuffle=True,
            random_state=outer_seed + fold_number,
        )
        inner_splits = list(inner_cv.split(X_outer_train, y_outer_train, groups=groups_outer_train))
        for inner_train, inner_valid in inner_splits:
            assert set(groups_outer_train[inner_train]).isdisjoint(
                set(groups_outer_train[inner_valid])
            )
            assert set(y_outer_train[inner_train]) == {0, 1}

        inner_oof_matrix = np.full((len(outer_train), len(base_models)), np.nan, dtype=float)
        outer_valid_matrix = np.full((len(outer_valid), len(base_models)), np.nan, dtype=float)

        for model_index, model in enumerate(base_models.values()):
            # 기본 모델 파라미터는 3단계 최적값으로 고정한다. 안쪽 CV는 앙상블
            # 가중치와 Stacking 학습에 필요한 OOF 확률을 만드는 데만 사용한다.
            inner_probability_matrix = cross_val_predict(
                clone(model),
                X_outer_train,
                y_outer_train,
                groups=groups_outer_train,
                cv=inner_splits,
                n_jobs=-1,
                method="predict_proba",
            )
            inner_oof_matrix[:, model_index] = inner_probability_matrix[:, 1]

            fitted_outer_model = clone(model).fit(X_outer_train, y_outer_train)
            outer_valid_matrix[:, model_index] = positive_class_probability(
                fitted_outer_model, X_train_raw.iloc[outer_valid]
            )

        assert np.isfinite(inner_oof_matrix).all()
        assert np.isfinite(outer_valid_matrix).all()
        base_oof[outer_valid] = outer_valid_matrix

        fold_weights, _ = greedy_ensemble_weights(y_outer_train, inner_oof_matrix)
        ges_oof[outer_valid] = outer_valid_matrix @ fold_weights

        stacking_model = LogisticRegression(
            C=0.1, max_iter=3000, solver="lbfgs", random_state=outer_seed
        ).fit(inner_oof_matrix, y_outer_train)
        stacking_oof[outer_valid] = positive_class_probability(stacking_model, outer_valid_matrix)

        fold_weight_rows.append(
            {
                "repeat": repeat_number,
                "outer_seed": outer_seed,
                "fold": fold_number,
                **dict(zip(base_model_names, fold_weights, strict=True)),
            }
        )
        print(f"반복 {repeat_number}/{len(OUTER_SEEDS)}, Fold {fold_number}/{OUTER_FOLDS} 완료")

    assert np.isfinite(base_oof).all()
    assert np.isfinite(ges_oof).all()
    assert np.isfinite(stacking_oof).all()
    repeat_base_oof.append(base_oof)
    repeat_ges_oof.append(ges_oof)
    repeat_stacking_oof.append(stacking_oof)

repeat_base_oof = np.stack(repeat_base_oof)
repeat_ges_oof = np.stack(repeat_ges_oof)
repeat_stacking_oof = np.stack(repeat_stacking_oof)
fold_ges_weights = pd.DataFrame(fold_weight_rows)
display(fold_ges_weights.round(3))
cv_seconds = perf_counter() - cv_started
print(f"TabICL 제외 반복 CV 시간: {cv_seconds:.2f}초")

반복 1/3, Fold 1/5 완료


반복 1/3, Fold 2/5 완료


반복 1/3, Fold 3/5 완료


반복 1/3, Fold 4/5 완료


반복 1/3, Fold 5/5 완료


...

반복 3/3, Fold 3/5 완료


반복 3/3, Fold 4/5 완료


반복 3/3, Fold 5/5 완료


,repeat,outer_seed,fold,LogisticRegression,MultinomialNB,ExtraTrees,CatBoost
0,1,1,1,0.00,0.32,0.28,0.40
1,1,1,2,0.00,0.12,0.68,0.20
2,1,1,3,0.00,0.12,0.40,0.48
3,1,1,4,0.00,0.36,0.00,0.64
4,1,1,5,0.00,0.28,0.04,0.68
5,2,11,1,0.00,0.36,0.08,0.56
6,2,11,2,0.00,0.36,0.00,0.64
7,2,11,3,0.00,0.60,0.00,0.40
8,2,11,4,0.00,0.16,0.00,0.84
9,2,11,5,0.24,0.16,0.00,0.60


TabICL 제외 반복 CV 시간: 28.83초


### 해석

바깥 검증 거래를 기본 모델·GES·Stacking 학습에서 모두 제외한다.
GES 가중치와 메타모델은 바깥 Train 안에서 만든 3-Fold OOF 확률만 본다.
시드 3개에서 각 거래의 검증 확률을 만든 뒤, 기존과 똑같이 반복별 전체 OOF를 평가한다.
폴드별 지표 평균을 쓰는 최근 RF 실험과 CV 집계 방법이 다르므로 그 CV 수치를 직접 섞어 순위를 매기지 않는다.

## 5. 전체 Train용 앙상블과 Test 확률 생성

In [6]:
final_fit_started = perf_counter()
# 반복 OOF 예측을 행별로 평균해 전체 Train에서 최종 가중치와 메타모델을 학습한다.
mean_base_oof = repeat_base_oof.mean(axis=0)
final_ges_weights, final_ges_history = greedy_ensemble_weights(y_train, mean_base_oof)
final_stacking_model = LogisticRegression(
    C=0.1, max_iter=3000, solver="lbfgs", random_state=OUTER_SEEDS[0]
).fit(mean_base_oof, y_train)

# 기본 모델은 전체 Train에 한 번씩만 다시 학습하고 모든 Test 세트의 확률을 보관한다.
fitted_base_models = {}
test_base_probability = {
    set_name: np.full((len(y_test), len(base_models)), np.nan, dtype=float)
    for set_name in X_test_raw_sets
}
for model_index, (model_name, model) in enumerate(base_models.items()):
    fitted_model = clone(model).fit(X_train_raw, y_train)
    fitted_base_models[model_name] = fitted_model
    for set_name, X_test in X_test_raw_sets.items():
        test_base_probability[set_name][:, model_index] = positive_class_probability(
            fitted_model, X_test
        )

final_weight_table = pd.DataFrame(
    {
        "model": base_model_names,
        "ges_weight": final_ges_weights,
        "selected_count": (final_ges_weights * GES_STEPS).round().astype(int),
    }
).sort_values("ges_weight", ascending=False, ignore_index=True)
display(final_weight_table.round(4))
display(final_ges_history.tail(10).round(6))
final_fit_seconds = perf_counter() - final_fit_started
print(f"최종 Train 학습 및 Test 확률 생성: {final_fit_seconds:.2f}초")

,model,ges_weight,selected_count
0,CatBoost,0.64,16
1,MultinomialNB,0.36,9
2,LogisticRegression,0.00,0
3,ExtraTrees,0.00,0


,step,selected_model,brier
15,16,CatBoost,0.192275
16,17,CatBoost,0.192281
17,18,MultinomialNB,0.192277
18,19,CatBoost,0.192276
19,20,MultinomialNB,0.192282
20,21,CatBoost,0.192276
21,22,CatBoost,0.192277
22,23,MultinomialNB,0.192278
23,24,CatBoost,0.192275
24,25,CatBoost,0.192278


최종 Train 학습 및 Test 확률 생성: 0.99초


### 해석

- Soft Voting은 네 모델의 확률을 같은 비중으로 평균한다.
- GES는 반복 OOF 평균에서 Brier를 줄이는 가중치를 고른다.
- Stacking은 이전과 똑같이 3회 반복의 5-Fold OOF를 행별 평균한 값으로 최종 메타 LR을 학습한다.
- Test는 가중치나 메타모델의 학습에 사용하지 않는다.

## 6. 단일 모델과 앙상블 최종 비교

In [7]:
candidate_cv_probabilities = {}
for model_index, model_name in enumerate(base_model_names):
    candidate_cv_probabilities[model_name] = repeat_base_oof[:, :, model_index]
candidate_cv_probabilities["SoftVoting_All"] = repeat_base_oof.mean(axis=2)
candidate_cv_probabilities["GES_25"] = repeat_ges_oof
candidate_cv_probabilities["Stacking_LR"] = repeat_stacking_oof

candidate_test_probabilities = {
    model_name: {
        set_name: probability_matrix[:, model_index]
        for set_name, probability_matrix in test_base_probability.items()
    }
    for model_index, model_name in enumerate(base_model_names)
}
candidate_test_probabilities["SoftVoting_All"] = {
    set_name: probability_matrix.mean(axis=1)
    for set_name, probability_matrix in test_base_probability.items()
}
candidate_test_probabilities["GES_25"] = {
    set_name: probability_matrix @ final_ges_weights
    for set_name, probability_matrix in test_base_probability.items()
}
candidate_test_probabilities["Stacking_LR"] = {
    set_name: positive_class_probability(final_stacking_model, probability_matrix)
    for set_name, probability_matrix in test_base_probability.items()
}

result_rows = []
test_results_by_candidate = {}
for candidate_name, repeated_probability in candidate_cv_probabilities.items():
    repeat_metrics = pd.DataFrame(
        [
            calculate_mask_set_averaged_metrics(
                y_train,
                probability,
                train_mask_set_labels,
            )
            for probability in repeated_probability
        ]
    )
    test_results = pd.DataFrame(
        [
            {
                "test_set": set_name,
                **calculate_metrics(y_test, probability),
            }
            for set_name, probability in candidate_test_probabilities[candidate_name].items()
        ]
    )
    test_results_by_candidate[candidate_name] = test_results
    test_summary = test_results[list(METRIC_NAMES)].agg(["mean", "std"]).T
    result_rows.append(
        {
            "model": candidate_name,
            "kind": "ensemble"
            if candidate_name in {"SoftVoting_All", "GES_25", "Stacking_LR"}
            else "single",
            "cv_brier_mean": repeat_metrics["brier"].mean(),
            "cv_brier_std": repeat_metrics["brier"].std(ddof=0),
            "cv_logloss_mean": repeat_metrics["logloss"].mean(),
            "cv_auc_mean": repeat_metrics["auc"].mean(),
            "cv_accuracy_mean": repeat_metrics["accuracy"].mean(),
            "cv_precision_mean": repeat_metrics["precision"].mean(),
            "cv_recall_mean": repeat_metrics["recall"].mean(),
            "cv_f1_mean": repeat_metrics["f1"].mean(),
            "cv_fp_mean": repeat_metrics["fp"].mean(),
            "cv_fn_mean": repeat_metrics["fn"].mean(),
            "test_brier_mean": test_summary.loc["brier", "mean"],
            "test_brier_std": test_summary.loc["brier", "std"],
            "test_auc_mean": test_summary.loc["auc", "mean"],
            "test_accuracy_mean": test_summary.loc["accuracy", "mean"],
            "test_precision_mean": test_summary.loc["precision", "mean"],
            "test_recall_mean": test_summary.loc["recall", "mean"],
            "test_f1_mean": test_summary.loc["f1", "mean"],
            "test_fp_mean": test_summary.loc["fp", "mean"],
            "test_fn_mean": test_summary.loc["fn", "mean"],
        }
    )

comparison = pd.DataFrame(result_rows).sort_values(
    ["cv_brier_mean", "cv_auc_mean", "cv_accuracy_mean"],
    ascending=[True, False, False],
    ignore_index=True,
)
ensemble_comparison = comparison.loc[comparison["kind"] == "ensemble"].reset_index(drop=True)
display(comparison.round(6))
print(f"반복 CV Brier 기준 전체 1순위: {comparison.iloc[0]['model']}")
print(f"반복 CV Brier 기준 앙상블 1순위: {ensemble_comparison.iloc[0]['model']}")

,model,kind,cv_brier_mean,cv_brier_std,cv_logloss_mean,cv_auc_mean,cv_accuracy_mean,cv_precision_mean,cv_recall_mean,cv_f1_mean,cv_fp_mean,cv_fn_mean,test_brier_mean,test_brier_std,test_auc_mean,test_accuracy_mean,test_precision_mean,test_recall_mean,test_f1_mean,test_fp_mean,test_fn_mean
0,Stacking_LR,ensemble,0.193801,0.002199,0.574461,0.767957,0.723962,0.695622,0.812159,0.749306,56.533333,29.866667,0.180932,0.006377,0.811479,0.732593,0.718896,0.772059,0.744102,20.6,15.5
1,SoftVoting_All,ensemble,0.194057,0.001892,0.575480,0.769378,0.724388,0.692807,0.822222,0.751926,58.000000,28.266667,0.180806,0.006200,0.812379,0.737037,0.718049,0.788235,0.751108,21.1,14.4
2,GES_25,ensemble,0.194687,0.002849,0.577998,0.768021,0.724281,0.690309,0.829769,0.753580,59.233333,27.066667,0.180980,0.006676,0.806212,0.740741,0.708769,0.825000,0.762161,23.1,11.9
3,CatBoost,single,0.194730,0.002340,0.578848,0.766774,0.723855,0.695429,0.812369,0.749255,56.600000,29.833333,0.186809,0.006452,0.790222,0.740741,0.702151,0.844118,0.766377,24.4,10.6
4,LogisticRegression,single,0.196919,0.001375,0.581281,0.762328,0.720021,0.696531,0.795807,0.742750,55.166667,32.466667,0.185205,0.006143,0.806080,0.713333,0.737183,0.673529,0.703239,16.5,22.2
5,MultinomialNB,single,0.196940,0.001893,0.582130,0.757841,0.715868,0.682415,0.825367,0.746959,61.166667,27.766667,0.181453,0.007900,0.800373,0.721481,0.720401,0.732353,0.725737,19.4,18.2
6,ExtraTrees,single,0.197144,0.002065,0.582248,0.762715,0.722258,0.692655,0.815094,0.748857,57.533333,29.400000,0.181914,0.005569,0.812028,0.742963,0.705081,0.842647,0.767612,24.0,10.7


반복 CV Brier 기준 전체 1순위: Stacking_LR
반복 CV Brier 기준 앙상블 1순위: Stacking_LR


### 해석 기준

- Train과 Test 모두 마스킹 10세트를 각각 평가한 뒤 평균하므로 AUC·Precision·F1도 같은 정의로 비교한다.
- `cv_*_mean`은 같은 모델을 세 가지 Fold 시드에서 평가한 평균이다. `cv_brier_std`가 크면 순위가 분할에 민감하다.
- Test 10세트의 표준편차는 신뢰구간이 아니라 어떤 4개 컬럼이 Unknown이 되는지에 따른 민감도다.
- Brier가 거의 같다면 AUC·Accuracy·Precision·Recall·FP·FN과 GES 가중치 안정성을 함께 보고, 운영 복잡도가 낮은 후보를 우선한다.
- Test 수치에 맞춰 가중치나 기본 모델 파라미터를 다시 바꾸지 않는다.

## 7. 기본 모델 확률의 다양성 확인

In [8]:
oof_probability_correlation = pd.DataFrame(
    mean_base_oof,
    columns=base_model_names,
).corr()
display(oof_probability_correlation.round(3))

expected_candidates = set(base_model_names) | {"SoftVoting_All", "GES_25", "Stacking_LR"}
assert set(candidate_cv_probabilities) == expected_candidates
assert set(candidate_test_probabilities) == expected_candidates
assert comparison["model"].nunique() == len(expected_candidates)
assert comparison.drop(columns=["model", "kind"]).notna().all().all()
assert np.allclose(final_ges_weights.sum(), 1.0)
assert (final_ges_weights >= 0).all()
assert all(
    len(results) == 10 and set(results["test_set"]) == set(X_test_raw_sets)
    for results in test_results_by_candidate.values()
)
assert all(
    np.isfinite(probability).all() and ((probability >= 0) & (probability <= 1)).all()
    for repeated_probability in candidate_cv_probabilities.values()
    for probability in repeated_probability
)


print("7개 후보의 반복 Group CV·확률 범위·Test 10세트 검증: 통과")

,LogisticRegression,MultinomialNB,ExtraTrees,CatBoost
LogisticRegression,1.000,0.970,0.965,0.948
MultinomialNB,0.970,1.000,0.946,0.911
ExtraTrees,0.965,0.946,1.000,0.966
CatBoost,0.948,0.911,0.966,1.000


7개 후보의 반복 Group CV·확률 범위·Test 10세트 검증: 통과


### 해석

상관이 낮은 모델은 단독 성능이 낮더라도 다른 모델의 오류를 보완할 수 있다.
따라서 TabICL 단독 순위가 낮았다는 사실만으로 제거 후 앙상블 성능이 좋아진다고 단정하지 않는다.

## 8. TabICL 포함 이전 결과와 비교

기준은 `deal_model_phase4.ipynb` 결과 셀 `3d2907a9`에 저장된 2026-08-26 실행 출력이다.
아래 숫자는 그 표의 소수점 6자리 값을 옮긴 것이며, TabICL 포함 실험을 이번에 다시 실행한 값은 아니다.
먼저 나머지 네 단일 모델이 같은 결과를 재현하는지 검사하고, 세 앙상블의 차이를 본다.

In [9]:
previous_results = pd.DataFrame(
    [
        {
            "model": "Stacking_LR",
            "cv_brier_mean": 0.194406,
            "test_brier_mean": 0.179131,
            "test_auc_mean": 0.817164,
            "test_accuracy_mean": 0.734074,
            "test_precision_mean": 0.720534,
            "test_recall_mean": 0.772059,
            "test_f1_mean": 0.745083,
            "test_fp_mean": 20.4,
            "test_fn_mean": 15.5,
        },
        {
            "model": "SoftVoting_All",
            "cv_brier_mean": 0.194633,
            "test_brier_mean": 0.177875,
            "test_auc_mean": 0.819557,
            "test_accuracy_mean": 0.740741,
            "test_precision_mean": 0.720975,
            "test_recall_mean": 0.792647,
            "test_f1_mean": 0.754755,
            "test_fp_mean": 20.9,
            "test_fn_mean": 14.1,
        },
        {
            "model": "CatBoost",
            "cv_brier_mean": 0.19473,
            "test_brier_mean": 0.186809,
            "test_auc_mean": 0.790222,
            "test_accuracy_mean": 0.740741,
            "test_precision_mean": 0.702151,
            "test_recall_mean": 0.844118,
            "test_f1_mean": 0.766377,
            "test_fp_mean": 24.4,
            "test_fn_mean": 10.6,
        },
        {
            "model": "GES_25",
            "cv_brier_mean": 0.195123,
            "test_brier_mean": 0.18098,
            "test_auc_mean": 0.806212,
            "test_accuracy_mean": 0.740741,
            "test_precision_mean": 0.708769,
            "test_recall_mean": 0.825,
            "test_f1_mean": 0.762161,
            "test_fp_mean": 23.1,
            "test_fn_mean": 11.9,
        },
        {
            "model": "LogisticRegression",
            "cv_brier_mean": 0.196919,
            "test_brier_mean": 0.185205,
            "test_auc_mean": 0.80608,
            "test_accuracy_mean": 0.713333,
            "test_precision_mean": 0.737183,
            "test_recall_mean": 0.673529,
            "test_f1_mean": 0.703239,
            "test_fp_mean": 16.5,
            "test_fn_mean": 22.2,
        },
        {
            "model": "MultinomialNB",
            "cv_brier_mean": 0.19694,
            "test_brier_mean": 0.181453,
            "test_auc_mean": 0.800373,
            "test_accuracy_mean": 0.721481,
            "test_precision_mean": 0.720401,
            "test_recall_mean": 0.732353,
            "test_f1_mean": 0.725737,
            "test_fp_mean": 19.4,
            "test_fn_mean": 18.2,
        },
        {
            "model": "ExtraTrees",
            "cv_brier_mean": 0.197144,
            "test_brier_mean": 0.181914,
            "test_auc_mean": 0.812028,
            "test_accuracy_mean": 0.742963,
            "test_precision_mean": 0.705081,
            "test_recall_mean": 0.842647,
            "test_f1_mean": 0.767612,
            "test_fp_mean": 24,
            "test_fn_mean": 10.7,
        },
    ]
).set_index("model")
current_results = comparison.set_index("model")[list(previous_results.columns)]

# 이전 표는 6자리 반올림이다. 단일 모델 재현이 안 되면 제거 효과 비교를 중단한다.
np.testing.assert_allclose(
    current_results.loc[base_model_names].to_numpy(),
    previous_results.loc[base_model_names].to_numpy(),
    atol=5.1e-7,
    rtol=0,
)
ensemble_names = ["SoftVoting_All", "GES_25", "Stacking_LR"]
before_after = pd.concat(
    {
        "TabICL 포함 · 이전 저장 출력": previous_results.loc[ensemble_names],
        "TabICL 제외 · 이번 실행": current_results.loc[ensemble_names],
    },
    names=["condition", "model"],
)
difference = current_results.loc[ensemble_names] - previous_results.loc[ensemble_names]
display(before_after.round(6))
display(difference.round(6))
print("네 단일 모델의 CV/Test가 이전 저장 결과와 일치: 통과")

cv_brier_mean  test_brier_mean  test_auc_mean  test_accuracy_mean  test_precision_mean  test_recall_mean  test_f1_mean  test_fp_mean  test_fn_mean
condition            model                                                                                                                                                             
TabICL 포함 · 이전 저장 출력 SoftVoting_All       0.194633         0.177875       0.819557            0.740741             0.720975          0.792647      0.754755          20.9          14.1
                     GES_25               0.195123         0.180980       0.806212            0.740741             0.708769          0.825000      0.762161          23.1          11.9
                     Stacking_LR          0.194406         0.179131       0.817164            0.734074             0.720534          0.772059      0.745083          20.4          15.5
TabICL 제외 · 이번 실행    SoftVoting_All       0.194057         0.180806       0.812379            0.737037             0.718049          0.788235      0.751108          21.1          14.4
                     GES_25               0.194687         0.180980       0.806212            0.740741             0.708769          0.825000      0.762161          23.1          11.9
                     Stacking_LR          0.193801         0.180932       0.811479            0.732593             0.718896          0.772059      0.744102          20.6          15.5

,cv_brier_mean,test_brier_mean,test_auc_mean,test_accuracy_mean,test_precision_mean,test_recall_mean,test_f1_mean,test_fp_mean,test_fn_mean
model,,,,,,,,,
SoftVoting_All,-0.000576,0.002931,-0.007178,-0.003704,-0.002926,-0.004412,-0.003647,0.2,0.3
GES_25,-0.000436,-0.000000,-0.000000,-0.000000,0.000000,0.000000,0.000000,0.0,0.0
Stacking_LR,-0.000605,0.001801,-0.005685,-0.001481,-0.001638,-0.000000,-0.000981,0.2,0.0


네 단일 모델의 CV/Test가 이전 저장 결과와 일치: 통과


### 해석

차이 표는 `TabICL 제외 − 포함`이다. AUC·Accuracy는 양수, Brier·FP/FN은 음수가 해당 지표의 개선이다.
성능과 속도는 구분한다. 제거 후 빨라졌더라도 예측 성능이 함께 좋아졌다고 해석하지 않는다.
Test 10세트는 같은 135건에 서로 다른 마스킹을 적용한 것이므로 1,350건의 새 거래가 아니다.

## 9. 단건 예측 시간

In [10]:
def candidate_probability(candidate, X):
    """이번 Train으로 학습한 모델만 사용해 Won 확률을 계산한다."""
    if candidate in fitted_base_models:
        return positive_class_probability(fitted_base_models[candidate], X)
    if candidate == "GES_25":
        active = np.flatnonzero(final_ges_weights > 0)
        return sum(
            final_ges_weights[index]
            * positive_class_probability(fitted_base_models[base_model_names[index]], X)
            for index in active
        )
    matrix = np.column_stack(
        [positive_class_probability(fitted_base_models[name], X) for name in base_model_names]
    )
    if candidate == "SoftVoting_All":
        return matrix.mean(axis=1)
    if candidate == "Stacking_LR":
        return positive_class_probability(final_stacking_model, matrix)
    raise ValueError(f"알 수 없는 후보: {candidate}")


known_row = {
    column: next(value for value in CATEGORY_VALUES[column] if value != "Unknown")
    for column in MODEL_FEATURE_NAMES
}
masked_row = {**known_row, **dict.fromkeys(MODEL_FEATURE_NAMES[:4], "Unknown")}
self_check_X = pd.DataFrame(
    [known_row, masked_row, dict.fromkeys(MODEL_FEATURE_NAMES, "Unknown")],
    columns=list(MODEL_FEATURE_NAMES),
)
timing_rows = []
for name in list(base_model_names) + ensemble_names:
    timing_input = self_check_X.iloc[[1]]
    candidate_probability(name, timing_input)
    durations = []
    for _ in range(20):
        started = perf_counter()
        candidate_probability(name, timing_input)
        durations.append((perf_counter() - started) * 1000)
    timing_rows.append(
        {
            "model": name,
            "warm_predict_mean_ms": np.mean(durations),
            "warm_predict_p95_ms": np.percentile(durations, 95),
        }
    )
timing_comparison = pd.DataFrame(timing_rows).set_index("model")
display(timing_comparison.round(3))

# 별도 예측 함수도 Test 평가에서 사용한 확률과 일치해야 한다.
for name in ensemble_names:
    first_set = next(iter(X_test_raw_sets))
    np.testing.assert_allclose(
        candidate_probability(name, X_test_raw_sets[first_set]),
        candidate_test_probabilities[name][first_set],
        atol=1e-12,
        rtol=1e-12,
    )

,warm_predict_mean_ms,warm_predict_p95_ms
model,,
LogisticRegression,0.949,1.078
MultinomialNB,0.942,1.031
ExtraTrees,5.263,5.608
CatBoost,0.107,0.129
SoftVoting_All,7.663,8.180
GES_25,1.253,1.768
Stacking_LR,7.917,9.003


### 해석

로컬 CPU에서 로딩을 마친 모델에 합성 입력 1건을 20회 요청한 warm 예측 시간이다.
AWS·네트워크·LLM 구조화·초기 로딩 시간은 포함하지 않는다.
이번에는 TabICL 포함 모델의 단건 지연을 재측정하지 않았으므로 정확한 배수 개선이나 서비스 지연의 원인 확정은 하지 않는다.

## 10. TabICL 제외 후보 저장

In [11]:
# 이 파일은 제거 실험의 후보 모음이다. 기존 RF·배포 Stacking 파일을 덮어쓰지 않는다.
bundle = {
    "schema_version": 1,
    "model_version": "deal-no-tabicl-v1",
    "base_model_order": base_model_names,
    "base_models": fitted_base_models,
    "stacking_model": final_stacking_model,
    "ges_weights": final_ges_weights,
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "model_feature_names": list(MODEL_FEATURE_NAMES),
    "category_values": {column: list(values) for column, values in CATEGORY_VALUES.items()},
    "target": {"Lost": 0, "Won": 1},
    "source_sha256": SOURCE_SHA256,
    "data_signature": training_data_signature(),
    "best_params": best_params,
    "training_scope": "train_split_only",
    "outer_seeds": OUTER_SEEDS,
    "outer_folds": OUTER_FOLDS,
    "inner_folds": INNER_FOLDS,
    "comparison": comparison,
    "previous_results": previous_results,
    "prediction_timing": timing_comparison,
}
NO_TABICL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(bundle, NO_TABICL_PATH)
restored = joblib.load(NO_TABICL_PATH)
assert restored["base_model_order"] == base_model_names
assert "TabICL" not in restored["base_models"]
assert restored["data_signature"] == training_data_signature()
restored_matrix = np.column_stack(
    [
        positive_class_probability(restored["base_models"][name], self_check_X)
        for name in restored["base_model_order"]
    ]
)
restored_probabilities = {
    "SoftVoting_All": restored_matrix.mean(axis=1),
    "GES_25": restored_matrix @ restored["ges_weights"],
    "Stacking_LR": positive_class_probability(restored["stacking_model"], restored_matrix),
}
for name, probability in restored_probabilities.items():
    np.testing.assert_allclose(
        probability, candidate_probability(name, self_check_X), atol=1e-12, rtol=1e-12
    )
    assert np.isfinite(probability).all() and ((probability >= 0) & (probability <= 1)).all()
print("3개 앙상블 저장 후 재로드 확률 일치: 통과")
print(f"후보 저장: backend/pipeline/artifacts/{NO_TABICL_PATH.name}")
print(f"파일 크기: {NO_TABICL_PATH.stat().st_size / 1024**2:.3f} MiB")

3개 앙상블 저장 후 재로드 확률 일치: 통과
후보 저장: backend/pipeline/artifacts/deal-no-tabicl-v1.joblib
파일 크기: 3.857 MiB


### 해석

현재 Train만 학습한 네 기본 모델과 GES 가중치·스태킹 메타모델을 함께 보관한다.
TabICL 체크포인트는 포함하지 않는다. 기존 백엔드 로더에 자동 적용되는 배포 파일은 아니다.
원본 영업 행·행별 정답·예측 배열은 새 후보 파일에 기록하지 않으며, 모델 파일은 Git 제외 대상이다.